Generate single-model responses for the configured benchmark and save responses with token usage.


In [ ]:
from pathlib import Path
import os
_repository_root = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'data').is_dir() and (p / 'experiments/clinical_protocols').is_dir()))
os.chdir(_repository_root / 'evaluations/geval_judges/source_protocols/baseline_deepseekv4')
import json
import hashlib
import re
import time
from typing import Dict, Any, List
import pandas as pd
import requests
from openai import OpenAI
from tqdm.auto import tqdm


In [ ]:
FOLDER_ID = ''
API_KEY = ''
LLM_API_URL = 'https://llm.api.cloud.yandex.net/foundationModels/v1/completion'
INPUT_CSV = 'llmfmed_dataset.csv'
OUTPUT_CSV = 'baseline_noroles_deepseekv4.csv'
BASELINE_MODEL_NAME = 'deepseekv4_default'
BASELINE_MODEL_URI = f'gpt://{FOLDER_ID}/deepseek-v4-flash/latest'
CASE_COMPONENT_COLUMNS = ['Vignette', 'Review of Systems', 'Past Medical History', 'Physical Examination', 'Test Results']
EMERGENCY_LABELS = ['Self-care appropriate', 'Requires non-emergent care', 'Requires emergent care']
SUBCLINICAL_LABELS = ['Cardiology', 'Dermatology', 'Emergency', 'Endocrinology', 'Gastroenterology', 'General Medicine', 'General Surgery', 'Infectious Disease', 'Nephrology', 'Neurology', 'Obstetrics and Gynecology', 'Ophthalmology', 'Orthopedics', 'Other departments', 'Otolaryngology (ENT)', 'Pediatrics', 'Psychiatry', 'Pulmonology', 'Urology']
REQUEST_TIMEOUT = 120
BASE_SLEEP_SECONDS = 2
MAX_RETRIES = 6
TEMPERATURE_PROTOCOL = 0.3
MAX_TOKENS_PROTOCOL = 3000
BASELINE_ROLE_NAME = 'Default'
BASELINE_ROLE_DESCRIPTION = 'You are a careful expert assistant. Follow the requested format exactly. Do not use information not grounded in the provided case.'

def detect_provider(model_uri: str) -> str:
    if 'aliceai-llm' in model_uri:
        return 'yandex_rest'
    return 'openai_compatible'
BASELINE_PROVIDER = detect_provider(BASELINE_MODEL_URI)
df = pd.read_csv(INPUT_CSV, sep=None, engine='python')
df = df.copy().reset_index(drop=True)
missing_case_columns = [col for col in CASE_COMPONENT_COLUMNS if col not in df.columns]
if missing_case_columns:
    raise ValueError(f'Missing case columns in dataset: {missing_case_columns}')

def build_case_text(row_data: Dict[str, Any]) -> str:
    parts: List[str] = []
    for column in CASE_COMPONENT_COLUMNS:
        value = row_data.get(column)
        if pd.isna(value):
            continue
        value_str = str(value).strip()
        if not value_str or value_str.lower() == 'nan':
            continue
        parts.append(f'{column}:\n{value_str}')
    return '\n\n'.join(parts)
df['case_text'] = df.apply(lambda row: build_case_text(row.to_dict()), axis=1)
print('Rows:', len(df))
print('Model:', BASELINE_MODEL_NAME, '->', BASELINE_MODEL_URI)
print('Provider:', BASELINE_PROVIDER)
print('Case columns:', CASE_COMPONENT_COLUMNS)


In [ ]:
session = requests.Session()
llm_cache: Dict[str, Dict[str, Any]] = {}
openai_client = OpenAI(api_key=API_KEY, base_url='https://llm.api.cloud.yandex.net/v1', project=FOLDER_ID)

def make_cache_key(provider: str, model_uri: str, system_text: str, user_text: str, temperature: float, max_tokens: int) -> str:
    payload = {'provider': provider, 'model_uri': model_uri, 'system_text': system_text, 'user_text': user_text, 'temperature': temperature, 'max_tokens': max_tokens}
    raw = json.dumps(payload, sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(raw.encode('utf-8')).hexdigest()

def _build_token_dict(prompt_tokens: int=0, completion_tokens: int=0, total_tokens: int=0) -> Dict[str, int]:
    return {'prompt_tokens': int(prompt_tokens or 0), 'completion_tokens': int(completion_tokens or 0), 'total_tokens': int(total_tokens or 0)}

def _extract_openai_text(response: Any) -> str:
    if not getattr(response, 'choices', None):
        return ''
    message = response.choices[0].message
    content = getattr(message, 'content', '')
    if isinstance(content, str):
        return content.strip()
    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, dict) and item.get('type') == 'text':
                parts.append(item.get('text', ''))
            elif hasattr(item, 'type') and getattr(item, 'type', None) == 'text':
                parts.append(getattr(item, 'text', ''))
        return '\n'.join([p for p in parts if p]).strip()
    return ''

def _extract_openai_usage(usage: Any) -> Dict[str, int]:
    if usage is None:
        return _build_token_dict()
    prompt_tokens = getattr(usage, 'prompt_tokens', None)
    completion_tokens = getattr(usage, 'completion_tokens', None)
    total_tokens = getattr(usage, 'total_tokens', None)
    if total_tokens is None:
        input_tokens = getattr(usage, 'input_tokens', None)
        output_tokens = getattr(usage, 'output_tokens', None)
        prompt_tokens = prompt_tokens if prompt_tokens is not None else input_tokens
        completion_tokens = completion_tokens if completion_tokens is not None else output_tokens
        total_tokens = int(prompt_tokens or 0) + int(completion_tokens or 0)
    return _build_token_dict(prompt_tokens, completion_tokens, total_tokens)

def _extract_yandex_usage(result: Dict[str, Any]) -> Dict[str, int]:
    usage = result.get('usage', {}) or {}
    prompt_tokens = usage.get('inputTextTokens', 0) or 0
    completion_tokens = usage.get('completionTokens', 0) or 0
    total_tokens = usage.get('totalTokens', 0) or prompt_tokens + completion_tokens
    return _build_token_dict(prompt_tokens, completion_tokens, total_tokens)

def call_yandex_llm(model_uri: str, system_text: str, user_text: str, temperature: float, max_tokens: int, timeout: int=REQUEST_TIMEOUT, retries: int=MAX_RETRIES) -> Dict[str, Any]:
    cache_key = make_cache_key('yandex_rest', model_uri, system_text, user_text, temperature, max_tokens)
    if cache_key in llm_cache:
        return {'text': llm_cache[cache_key]['text'], 'prompt_tokens': 0, 'completion_tokens': 0, 'total_tokens': 0, 'cached': True}
    headers = {'Content-Type': 'application/json', 'Authorization': f'Api-Key {API_KEY}'}
    payload = {'modelUri': model_uri, 'completionOptions': {'stream': False, 'temperature': temperature, 'maxTokens': max_tokens}, 'messages': [{'role': 'system', 'text': system_text}, {'role': 'user', 'text': user_text}]}
    last_error_text = ''
    for attempt in range(retries):
        try:
            response = session.post(LLM_API_URL, headers=headers, json=payload, timeout=timeout)
        except requests.RequestException as e:
            last_error_text = str(e)
            time.sleep(BASE_SLEEP_SECONDS * 2 ** attempt)
            continue
        if response.status_code == 200:
            data = response.json()
            result = data.get('result', {}) or {}
            raw_text = result.get('alternatives', [{}])[0].get('message', {}).get('text', '').strip()
            usage = _extract_yandex_usage(result)
            llm_cache[cache_key] = {'text': raw_text}
            return {'text': raw_text, **usage, 'cached': False}
        last_error_text = f'HTTP {response.status_code}: {response.text}'
        if response.status_code in (429, 500, 502, 503, 504):
            time.sleep(BASE_SLEEP_SECONDS * 2 ** attempt)
            continue
        raise RuntimeError(last_error_text)
    raise RuntimeError(f'Yandex REST call failed after {retries} retries. Last error: {last_error_text}')

def call_openai_llm(model_uri: str, system_text: str, user_text: str, temperature: float, max_tokens: int, retries: int=MAX_RETRIES) -> Dict[str, Any]:
    cache_key = make_cache_key('openai_compatible', model_uri, system_text, user_text, temperature, max_tokens)
    if cache_key in llm_cache:
        return {'text': llm_cache[cache_key]['text'], 'prompt_tokens': 0, 'completion_tokens': 0, 'total_tokens': 0, 'cached': True}
    last_error_text = ''
    for attempt in range(retries):
        try:
            response = openai_client.chat.completions.create(model=model_uri, messages=[{'role': 'system', 'content': system_text}, {'role': 'user', 'content': user_text}], temperature=temperature, max_tokens=max_tokens)
            raw_text = _extract_openai_text(response)
            usage = _extract_openai_usage(getattr(response, 'usage', None))
            llm_cache[cache_key] = {'text': raw_text}
            return {'text': raw_text, **usage, 'cached': False}
        except Exception as e:
            last_error_text = str(e)
            time.sleep(BASE_SLEEP_SECONDS * 2 ** attempt)
            continue
    raise RuntimeError(f'OpenAI-compatible call failed after {retries} retries. Last error: {last_error_text}')

def call_llm(provider: str, model_uri: str, system_text: str, user_text: str, temperature: float, max_tokens: int) -> Dict[str, Any]:
    if provider == 'yandex_rest':
        return call_yandex_llm(model_uri, system_text, user_text, temperature, max_tokens)
    if provider == 'openai_compatible':
        return call_openai_llm(model_uri, system_text, user_text, temperature, max_tokens)
    raise ValueError(f'Unknown provider: {provider}')


In [ ]:
def build_stage1_system_prompt(role_name: str, role_description: str) -> str:
    return f'You are acting as {role_name}.\n{role_description}\n\nYou are a medical expert.\nAt this stage, you need to evaluate a specific clinical case and provide a final structured protocol based on the case.\n\nUse only information explicitly present in the case. If a detail is absent, write "Not specified"; if a field is unnecessary, write "None".\nReason carefully from the case. Do not invent symptoms or findings absent from the case.\n'.strip()

def build_stage1_user_prompt(case_text: str, author_id: str) -> str:
    emergency_list = json.dumps(EMERGENCY_LABELS)
    subclinical_list = json.dumps(SUBCLINICAL_LABELS)
    return f'Read the clinical case and write one medical protocol.\n\nClinical case:\n<<<CASE>>>\n{case_text}\n<<<END CASE>>>\n\nWrite the protocol in plain text. Same field labels and order:\n\nAge:\nSex:\nComplaint:\nAnalysis:\nRequired diagnostic equipment/tests:\nConclusion:\nEmergency type:\nSubclinical specialization:\nRecommendations:\nNotes:\nProtocol Author: {author_id}\n\nField instructions:\nAge: copy from the case if stated; otherwise write "Not specified".\nSex: copy from the case if stated; otherwise write "Not specified".\nComplaint: briefly summarize the presenting problem from the case.\nAnalysis: explain the clinical reasoning, the strongest supporting facts, and the main relevant differential considerations.\nRequired diagnostic equipment/tests: mention only studies or equipment that are actually needed next; if nothing additional is needed, write "None".\nConclusion: give a concise clinical synthesis on the case.\nEmergency type: based on your analysis of this case, emergency type must equal exactly one of {emergency_list}.\nSubclinical specialization: based on your analysis of this case, subclinical specialization must equal exactly one of {subclinical_list}.\nRecommendations: provide practical, case-grounded next steps or management advice. Explicitly address the additional considerations.\nNotes: include only essential caveats; otherwise write "None".\n'.strip()

def strip_author_section(protocol_text: str) -> str:
    if protocol_text is None:
        return ''
    text = str(protocol_text).strip()
    parts = re.split('(?im)^\\s*Protocol Author\\s*:', text, maxsplit=1)
    return parts[0].rstrip()


In [ ]:
def run_baseline_pipeline(input_df: pd.DataFrame) -> pd.DataFrame:
    rows: List[Dict[str, Any]] = []
    system_text = build_stage1_system_prompt(BASELINE_ROLE_NAME, BASELINE_ROLE_DESCRIPTION)
    for _, row in tqdm(input_df.iterrows(), total=len(input_df), desc='Baseline pipeline'):
        row_data = row.to_dict()
        case_text = row_data['case_text']
        user_text = build_stage1_user_prompt(case_text, BASELINE_MODEL_NAME)
        response = call_llm(provider=BASELINE_PROVIDER, model_uri=BASELINE_MODEL_URI, system_text=system_text, user_text=user_text, temperature=TEMPERATURE_PROTOCOL, max_tokens=MAX_TOKENS_PROTOCOL)
        full_protocol = response['text'].strip()
        row_data['baseline_protocol'] = full_protocol
        row_data['baseline_protocol_anonymous'] = strip_author_section(full_protocol)
        row_data['baseline_model_id'] = BASELINE_MODEL_NAME
        row_data['baseline_model_uri'] = BASELINE_MODEL_URI
        row_data['baseline_provider'] = BASELINE_PROVIDER
        row_data['baseline_role_name'] = BASELINE_ROLE_NAME
        row_data['baseline_role_description'] = BASELINE_ROLE_DESCRIPTION
        row_data['prompt_tokens'] = response['prompt_tokens']
        row_data['completion_tokens'] = response['completion_tokens']
        row_data['total_tokens'] = response['total_tokens']
        row_data['cached'] = response['cached']
        rows.append(row_data)
    return pd.DataFrame(rows)
result_df = run_baseline_pipeline(df)
print(result_df.shape)
result_df.head(5)


In [ ]:
def build_token_summary(result_df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame([{'model_id': BASELINE_MODEL_NAME, 'model_uri': BASELINE_MODEL_URI, 'provider': BASELINE_PROVIDER, 'role_name': BASELINE_ROLE_NAME, 'prompt_tokens': int(result_df['prompt_tokens'].fillna(0).sum()), 'completion_tokens': int(result_df['completion_tokens'].fillna(0).sum()), 'total_tokens': int(result_df['total_tokens'].fillna(0).sum()), 'n_cases': len(result_df), 'n_cached': int(result_df['cached'].fillna(False).sum())}])
token_summary_df = build_token_summary(result_df)
token_summary_df


In [ ]:
result_df_part = result_df.copy()
result_df_part['table_source'] = 'result_df'
token_summary_part = token_summary_df.copy()
token_summary_part['table_source'] = 'token_summary_df'
combined_output_df = pd.concat([result_df_part, token_summary_part], ignore_index=True, sort=False)
combined_output_df.to_csv(OUTPUT_CSV, sep=';', index=False, encoding='utf-8-sig')
print('Saved:', OUTPUT_CSV)
print('Rows (result_df):', len(result_df))
print('Total tokens:', token_summary_df['total_tokens'].iloc[0])
